# Data Detective: Ventas

Vamos a revisar el archivo `ventas.csv` como si fuéramos detectives:
primero lo exploramos, después buscamos errores escondidos, y al final
respondemos algunas preguntas sobre el negocio.

**Columnas del archivo:** Fecha, Producto, Region, Vendedor, Cantidad, Precio, Ventas

**Preguntas del reto:**
1. ¿Qué región genera más ingresos?
2. ¿Qué producto vende más?
3. ¿Cuáles son los 5 vendedores con más ventas?
4. ¿Cuántos días no se registró ninguna venta?
5. ¿El monto de "Ventas" siempre coincide con Cantidad x Precio?
6. ¿Hay alguna región que nunca vendió alguno de los productos?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("ventas.csv")
df.head()

## Estructura del archivo

Cuántas filas y columnas tiene, y de qué tipo es cada columna.

In [ ]:
print("Filas y columnas:", df.shape)
print()
print(df.dtypes)

## Buscar errores ocultos

Antes de contestar cualquier pregunta hay que auditar los datos: revisar
filas repetidas, datos vacíos, números que no tienen sentido (cantidades o
precios negativos) y que la columna Ventas sí sea Cantidad x Precio.

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
print()
print("Datos vacíos por columna:")
print(df.isna().sum())
print()
print("Filas con Cantidad menor o igual a 0:", len(df[df["Cantidad"] <= 0]))
print("Filas con Precio menor o igual a 0:", len(df[df["Precio"] <= 0]))

In [ ]:
# Ventas debería ser siempre Cantidad x Precio, lo comprobamos
diferencia = (df["Ventas"] - df["Cantidad"] * df["Precio"]).abs()
print("Filas donde Ventas no coincide con Cantidad x Precio:", len(df[diferencia > 0.01]))

No encontramos filas duplicadas, ni datos vacíos, ni cantidades o precios
negativos, y la columna Ventas siempre coincide con Cantidad x Precio. El
archivo está limpio, así que podemos contestar las preguntas con confianza.

## Pregunta 1: ¿Qué región genera más ingresos?

In [ ]:
ingresos_region = df.groupby("Region")["Ventas"].sum().sort_values(ascending=False)
print(ingresos_region)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(ingresos_region.index, ingresos_region.values, color="#2a78d6")
plt.title("Ingresos totales por región")
plt.ylabel("Ventas ($)")
plt.show()

## Pregunta 2: ¿Qué producto vende más?

Aquí hay que fijarse en dos cosas distintas: cuántas *unidades* se vendieron
de cada producto, y cuánto *dinero* generó cada producto. No siempre es lo
mismo.

In [ ]:
unidades_producto = df.groupby("Producto")["Cantidad"].sum().sort_values(ascending=False)
ingresos_producto = df.groupby("Producto")["Ventas"].sum().sort_values(ascending=False)

print("Unidades vendidas por producto:")
print(unidades_producto)
print()
print("Ingresos por producto:")
print(ingresos_producto)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(unidades_producto.index, unidades_producto.values, color="#2a78d6")
ax1.set_title("Unidades vendidas")
ax1.tick_params(axis="x", rotation=45)

ax2.bar(ingresos_producto.index, ingresos_producto.values, color="#eb6834")
ax2.set_title("Ingresos ($)")
ax2.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

Mouse es el producto que más *unidades* vende, pero Laptop es el que más
*dinero* genera, por mucha diferencia: se vende poco pero es muy caro.
Ahí está el hallazgo: vender mucho no siempre es lo mismo que ganar más.

## Pregunta 3: ¿Cuáles son los 5 vendedores con más ventas?

In [ ]:
ventas_vendedor = df.groupby("Vendedor")["Ventas"].sum().sort_values(ascending=False)
top_5 = ventas_vendedor.head(5)
print(top_5)

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(top_5.index, top_5.values, color="#1a9c74")
plt.title("Top 5 vendedores por ventas totales")
plt.ylabel("Ventas ($)")
plt.show()

## Pregunta 4: ¿Cuántos días no se registró ninguna venta?

In [ ]:
df["Fecha"] = pd.to_datetime(df["Fecha"])

todos_los_dias = pd.date_range(df["Fecha"].min(), df["Fecha"].max())
dias_con_ventas = df["Fecha"].unique()

print("Días en el rango de fechas:", len(todos_los_dias))
print("Días con al menos una venta:", len(dias_con_ventas))
print("Días sin ninguna venta:", len(todos_los_dias) - len(dias_con_ventas))

## Pregunta 5: ¿el monto de Ventas siempre coincide con Cantidad x Precio?

Ya lo revisamos en la sección de errores ocultos: la diferencia fue 0 en
todas las filas, así que sí coincide siempre. No hay montos inventados.

## Pregunta 6: ¿Hay alguna región que nunca vendió algún producto?

In [ ]:
productos = df["Producto"].unique()

for region in df["Region"].unique():
    productos_region = df[df["Region"] == region]["Producto"].unique()
    faltantes = [p for p in productos if p not in productos_region]
    print(region, "- productos que nunca vendió:", faltantes if faltantes else "ninguno")

## Conclusión

**Pregunta:** ¿vender más unidades siempre significa generar más ingresos?

**Hipótesis:** no necesariamente, porque cada producto tiene un precio muy
distinto.

**Estadística:** Mouse es el producto más vendido en unidades (668), pero
Laptop genera más ingresos con solo 300 unidades vendidas, porque su precio
promedio es de casi $15,000 contra los $450 del Mouse.

**Gráfica:** las gráficas de barras de la Pregunta 2 muestran justo eso: el
producto más alto en unidades no es el más alto en ingresos.

**Hallazgo:** la región Centro genera más ingresos que las demás, y los
vendedores Sofia y Ana están arriba del resto en ventas totales.

**Errores ocultos:** después de auditar el archivo no encontramos filas
duplicadas, datos vacíos, cantidades o precios negativos, ni desajustes
entre Ventas y Cantidad x Precio. El archivo está limpio.

**Conclusión final:** confirmamos la hipótesis. El producto que más se
vende no es el que más dinero deja — antes de decidir qué producto
impulsar, hay que ver los ingresos, no solo las unidades vendidas.